In [ ]:
#a. Quitar tildes de los campos de texto.
def quitar_tildes(texto):
    if pd.isna(texto):
        return texto
    return "".join(c for c in unicodedata.normalize("NFD", texto) if unicodedata.category(c) != "Mn")

In [ ]:
#b. Función de mapeo de géneros de libros.
def mapear_genero_libro(genero):
    for clave, valores in map_genero.items():
        if genero in valores:
            return clave
    return genero  # si no matchea nada, lo dejo sin agrupar (para detectarlo)

In [ ]:
#c. Función para calcular rating promedio dejando afuera el registro actual (leave-one-out, evita leakage).
def loo_mean(df, group_cols, target_col="rating"):
    grp = df.groupby(group_cols)[target_col]
    return (grp.transform("sum") - df[target_col]) / (grp.transform("count") - 1)

In [ ]:
#d. Replica del feature engineering sobre dataframe a predecir.
def feature_engineering_test(id_lector, id_libros):
    
    #1. Creo las combinaciones lector-libro candidato.
    #print("Creo las combinaciones id_lector-id_libro.")
    df_test_features = pd.DataFrame({
        "id_lector": id_lector,
        "id_libro": id_libros
    })
    
    #2. Mergeo características del lector.
    #print("Mergeo características del lector.")
    df_test_features = df_test_features.merge(
        caract_lector,
        on="id_lector",
        how="left"
    )
    
    #3. Mergeo características del libro.
    #print("Mergeo características del libro.")
    df_test_features = df_test_features.merge(
        caract_libros,
        on="id_libro",
        how="left"
    )

    #4. Mergeo afinidad lector-autor.
    #print("Mergeo características de afinidad lector-autor.")
    df_test_features = df_test_features.merge(
        afinidad_lector_autor_test,
        on=["id_lector", "autor"],
        how="left"
    )
    
    #5. Completo variables de afinidad lector-autor.
    
    # Si nunca interactuó con el autor, cantidad = 0.
    df_test_features["n_interacciones_lector_autor"] = (
        df_test_features["n_interacciones_lector_autor"]
        .fillna(0)
    )
    
    # Si nunca interactuó con el autor, rating = media global.
    df_test_features["rating_prom_id_lector_autor"] = (
        df_test_features["rating_prom_id_lector_autor"]
        .fillna(media_global)
    )

    #6. Mergeo afinidad lector-género.
    #print("Mergeo características de afinidad lector-género.")
    df_test_features = df_test_features.merge(
        afinidad_lector_genero_test,
        on=["id_lector", "genero_libro_agrupado"],
        how="left"
    )
    
    #7. Completo variables de afinidad lector-género.
    
    # Si nunca interactuó con el género, cantidad = 0.
    df_test_features["n_interacciones_lector_genero"] = (
        df_test_features["n_interacciones_lector_genero"]
        .fillna(0)
    )
    
    # Si nunca interactuó con el género, rating = media global.
    df_test_features[
        "rating_prom_id_lector_genero_libro_agrupado"
    ] = (
        df_test_features[
            "rating_prom_id_lector_genero_libro_agrupado"
        ]
        .fillna(media_global)
    )
    
    #8. Features temporales para candidatos.
    #print("Calculo características temporales de la interacción.")
    
    #df_test_features["edad_al_interactuar"] = (
    #    anio_actual - df_test_features["nacimiento"]
    #)

    # No existe una interacción real todavía.
    #df_test_features["dias_transcurridos_interaccion"] = 0 # sacar ya que en test todo va a ser 0.

    #df_test_features["anios_transcurridos_edicion"] = (
    #    anio_actual - df_test_features["anio_edicion"]
    #)

    #df_test_features["antiguedad_libro_hoy"] = ( 
    #    anio_actual - df_test_features["anio_edicion"] # sacar ya que en test todo va a ser a anios_transcurridos_edicion.
    #)
    
    return df_test_features

In [ ]:
#e. Armo las tablas de referencia para predecir sobre pares (lector, libro) que no existen.
# Acá NO va LOO. El LOO es para entrenar, donde cada fila tiene su propio rating
# como target. Acá el par todavía no existe, no hay nada que excluir.
def armar_tablas_referencia(df_base, df_libros, df_lectores, verbose=True):
    """
    Devuelve (media_global, caract_lector, caract_libros, afinidad_autor, afinidad_genero).
    df_base es la base de historial: df_train en validación, train+test en entrega.
    """
    #a. Media global del rating en la base.
    media_global = df_base["rating"].mean()

    #b. Características del lector.
    #i. Columnas base.
    x_caract_lector_base = [
        "id_lector",
        #"nombre",
        #"vive_en",
        "nacimiento",
        "ciudad",
        "pais"
    ]
    caract_lector_base = df_lectores[x_caract_lector_base].drop_duplicates("id_lector")

    #ii. Dummies desde df_lectores (tiene TODOS los lectores), no desde df_base
    # (solo tiene los que interactuaron). Si no, todo lector sin historial queda en NaN.
    cols_a_dummificar_lector = [
        "genero_persona",
        #"pais_agrupado",
    ]
    caract_lector_dummies = pd.get_dummies(
        df_lectores[["id_lector"] + cols_a_dummificar_lector].drop_duplicates("id_lector"),
        columns=cols_a_dummificar_lector
    )

    #iii. Alineo con las columnas exactas que vio el modelo en train:
    # si aparece una categoría que train no tenía, la descarto; si falta una que espera, la creo en 0.
    cols_dum_lector = [c for c in df_base.columns if c.startswith((
        "genero_persona_",
        #"pais_agrupado_",
    ))]
    caract_lector_dummies = caract_lector_dummies.reindex(
        columns=["id_lector"] + cols_dum_lector, fill_value=0
    )

    caract_lector = caract_lector_base.merge(caract_lector_dummies, how="left", on="id_lector")

    #iv. Red de seguridad: si algún lector no matcheó en el merge, las dummies quedan en 0.
    caract_lector[cols_dum_lector] = caract_lector[cols_dum_lector].fillna(0)

    #v. Frecuencia del lector ----> Se calcula sobre todo df_base, sin LOO.
    caract_lector["frecuencia_lector"] = (
        caract_lector["id_lector"]
        .map(df_base.groupby("id_lector").size())
        .fillna(0)
    )

    #vi. Rating promedio del lector ----> Se calcula sobre todo df_base, sin LOO.
    caract_lector["rating_prom_id_lector"] = (
        caract_lector["id_lector"]
        .map(df_base.groupby("id_lector")["rating"].mean())
        .fillna(media_global)
    )

    #vii. Cantidad de autores distintos que leyó ----> Se calcula sobre todo df_base, sin LOO.
    caract_lector["n_autores_distintos_lector"] = (
        caract_lector["id_lector"]
        .map(df_base.groupby("id_lector")["autor"].nunique())
        .fillna(0)
    )

    #viii. Cantidad de géneros distintos que leyó ----> Se calcula sobre todo df_base, sin LOO.
    caract_lector["n_generos_distintos_lector"] = (
        caract_lector["id_lector"]
        .map(df_base.groupby("id_lector")["genero_libro_agrupado"].nunique())
        .fillna(0)
    )

    #c. Características del libro.
    #i. Columnas base.
    x_caract_libros_base = [
        "id_libro",
        "autor",
        "genero_libro_agrupado",
        #"editorial",
        "anio_edicion",
        #"isbn",
        #"resumen"
    ]
    caract_libros_base = df_libros[x_caract_libros_base].drop_duplicates("id_libro")

    #ii. Dummies desde df_libros (tiene TODO el catálogo), no desde df_base
    # (solo tiene los libros que alguien leyó). Si no, todo libro sin interacciones queda en NaN.
    cols_a_dummificar_libro = [
        "genero_libro_agrupado",
        #"editorial_agrupada",
    ]
    caract_libros_dummies = pd.get_dummies(
        df_libros[["id_libro"] + cols_a_dummificar_libro].drop_duplicates("id_libro"),
        columns=cols_a_dummificar_libro
    )

    #iii. Alineo con las columnas exactas que vio el modelo en train.
    cols_dum_libro = [c for c in df_base.columns if c.startswith((
        "genero_libro_agrupado_",
        #"editorial_agrupada_",
    ))]
    caract_libros_dummies = caract_libros_dummies.reindex(
        columns=["id_libro"] + cols_dum_libro, fill_value=0
    )

    caract_libros = caract_libros_base.merge(caract_libros_dummies, how="left", on="id_libro")

    #iv. Red de seguridad: si algún libro no matcheó en el merge, las dummies quedan en 0.
    caract_libros[cols_dum_libro] = caract_libros[cols_dum_libro].fillna(0)

    #v. Frecuencia del libro ----> Se calcula sobre todo df_base, sin LOO.
    caract_libros["frecuencia_libro"] = (
        caract_libros["id_libro"]
        .map(df_base.groupby("id_libro").size())
        .fillna(0)
    )

    #vi. Cantidad de lectores distintos del autor ----> Se calcula sobre todo df_base, sin LOO.
    caract_libros["n_lectores_distintos_autor"] = (
        caract_libros["autor"]
        .map(df_base.groupby("autor")["id_lector"].nunique())
        .fillna(0)
    )

    #vii. Rating promedio del libro ----> Se calcula sobre todo df_base, sin LOO.
    # PENDIENTE: probar shrinkage hacia la media global (un libro con 2 lecturas no tiene
    # la misma evidencia que uno con 200). Si lo hago, va también en el LOO del notebook 2,
    # o la columna significa cosas distintas al entrenar y al predecir.
    caract_libros["rating_prom_id_libro"] = (
        caract_libros["id_libro"]
        .map(df_base.groupby("id_libro")["rating"].mean())
        .fillna(media_global)
    )

    #viii. Rating promedio del autor ----> Se calcula sobre todo df_base, sin LOO.
    caract_libros["rating_prom_autor"] = (
        caract_libros["autor"]
        .map(df_base.groupby("autor")["rating"].mean())
        .fillna(media_global)
    )

    #ix. Rating promedio del género ----> Se calcula sobre todo df_base, sin LOO.
    caract_libros["rating_prom_genero"] = (
        caract_libros["genero_libro_agrupado"]
        .map(df_base.groupby("genero_libro_agrupado")["rating"].mean())
        .fillna(media_global)
    )

    #d. Afinidades.
    #i. Afinidad lector-autor ----> Se calcula sobre todo df_base, sin LOO.
    afinidad_autor = (
        df_base
        .groupby(["id_lector", "autor"])
        .agg(
            rating_prom_id_lector_autor=("rating", "mean"),
            n_interacciones_lector_autor=("rating", "count")
        )
        .reset_index()
    )

    #ii. Afinidad lector-género ----> Se calcula sobre todo df_base, sin LOO.
    afinidad_genero = (
        df_base
        .groupby(["id_lector", "genero_libro_agrupado"])
        .agg(
            rating_prom_id_lector_genero_libro_agrupado=("rating", "mean"),
            n_interacciones_lector_genero=("rating", "count")
        )
        .reset_index()
    )

    #e. Comprobaciones.
    if verbose:
        nulos_lector = caract_lector.drop(columns=["ciudad", "pais"]).isna().sum()
        nulos_libros = caract_libros.isna().sum()

        print("Nulos caract_lector:", nulos_lector[nulos_lector > 0].to_dict() or "ninguno")
        print("Nulos caract_libros:", nulos_libros[nulos_libros > 0].to_dict() or "ninguno")

        print(
            "Duplicados (lector / libro / lector-autor / lector-genero):",
            caract_lector["id_lector"].duplicated().sum(),
            caract_libros["id_libro"].duplicated().sum(),
            afinidad_autor.duplicated(["id_lector", "autor"]).sum(),
            afinidad_genero.duplicated(["id_lector", "genero_libro_agrupado"]).sum()
        )

        print(
            f"Lectores: {len(caract_lector)} | Libros: {len(caract_libros)} | "
            f"Libros sin interacciones en la base: {(caract_libros['frecuencia_libro'] == 0).sum()}"
        )

    return media_global, caract_lector, caract_libros, afinidad_autor, afinidad_genero


In [ ]:
#f. Me devuelve por id_lector todos los id_libros que son candidatos a ser recomendados.
def retrieval(id_lector):
    """Retorna todos los libros que se pueden recomendar a id_lector."""
    
    libros_leidos = leidos_por_lector.get(id_lector, set())
    libros_no_leidos = list(set(todos_los_libros) - libros_leidos)
    
    return libros_no_leidos

In [ ]:
#g. Devuelvo un score predicho para cada id_libro candidato de un id_lector.
def ranking(df_features, features_modelo, modelo):
    """Predice el rating de cada libro candidato a partir de sus features ya calculadas."""

    #1. Selecciono exactamente las variables que utiliza el modelo.
    X = df_features[features_modelo]

    #2. Predigo el rating para cada libro candidato.
    predicciones = modelo.predict(X)

    #3. Devuelvo un diccionario {id_libro: rating_predicho}.
    return dict(zip(df_features["id_libro"], predicciones))

In [ ]:
#h. Función para limpiar el año de edición.
def limpiar_anio_edicion(df, anio_actual):
    """
    Limpia y completa la columna 'anio_edicion'.

    Pasos:
    1. Extrae años de 4 dígitos.
    2. Convierte valores inválidos a NaN.
    3. Elimina años fuera del rango [1800, anio_actual].
    4. Imputa faltantes con el promedio del título.
    5. Imputa los restantes con el promedio de la editorial.
    6. Imputa los restantes con la mediana general.
    7. Convierte la columna a int.
    """

    #1. Extraer año válido y convertir a numérico
    df["anio_edicion"] = (
        df["anio_edicion"]
        .astype(str)
        .str.extract(r"(\d{4})")[0]
    )

    df["anio_edicion"] = pd.to_numeric(
        df["anio_edicion"],
        errors="coerce"
    )

    print(
        f"Nulos tras extraer año válido: "
        f"{df['anio_edicion'].isnull().sum()}"
    )

    #2. Nulificar años fuera de rango
    mask_fuera_rango = (
        (df["anio_edicion"] < 1800) |
        (df["anio_edicion"] > anio_actual)
    )

    print(
        f"Años fuera de rango [1800, {anio_actual}] "
        f"nulificados: {mask_fuera_rango.sum()}"
    )

    df.loc[mask_fuera_rango, "anio_edicion"] = np.nan

    #3. Imputar por promedio del título
    promedio_por_titulo = (
        df.groupby("titulo")["anio_edicion"]
        .transform("mean")
        .round()
    )

    df["anio_edicion"] = df["anio_edicion"].fillna(
        promedio_por_titulo
    )

    print(
        f"Nulos tras imputar por promedio de título: "
        f"{df['anio_edicion'].isnull().sum()}"
    )

    #4. Imputar por promedio de editorial
    promedio_por_editorial = (
        df.groupby("editorial")["anio_edicion"]
        .transform("mean")
        .round()
    )

    df["anio_edicion"] = df["anio_edicion"].fillna(
        promedio_por_editorial
    )

    print(
        f"Nulos tras imputar por promedio de editorial: "
        f"{df['anio_edicion'].isnull().sum()}"
    )

    #5. Imputar restantes con mediana general
    mediana_general = round(df["anio_edicion"].median())

    df["anio_edicion"] = df["anio_edicion"].fillna(
        mediana_general
    )

    print(
        f"Nulos tras imputar por mediana general: "
        f"{df['anio_edicion'].isnull().sum()}"
    )

    #6. Convertir a entero
    df["anio_edicion"] = df["anio_edicion"].astype(int)

    return df

In [ ]:
#i. Limpiar la fecha de nacimiento.
def limpiar_nacimiento(df, anio_actual):
    """
    Limpia y completa la columna 'nacimiento'.

    Pasos:
    1. Convierte la columna a numérico.
    2. Nulifica años fuera del rango [1900, anio_actual].
    3. Imputa faltantes con el promedio de nacimiento por nombre.
    4. Imputa los restantes con el promedio global.
    5. Convierte la columna a entero.
    """

    #1. Forzar a numérico
    df["nacimiento"] = pd.to_numeric(
        df["nacimiento"],
        errors="coerce"
    )

    print(
        f"Nulos tras forzar a numérico: "
        f"{df['nacimiento'].isnull().sum()}"
    )

    #2. Nulificar nacimientos fuera de rango
    mask_fuera_rango = (
        (df["nacimiento"] < 1900) |
        (df["nacimiento"] > anio_actual)
    )

    print(
        f"Nacimientos fuera de rango [1900, {anio_actual}] "
        f"nulificados: {mask_fuera_rango.sum()}"
    )

    df.loc[mask_fuera_rango, "nacimiento"] = np.nan

    #3. Imputar por promedio del nombre
    promedio_por_nombre = (
        df.groupby("nombre")["nacimiento"]
        .transform("mean")
        .round()
    )

    df["nacimiento"] = df["nacimiento"].fillna(
        promedio_por_nombre
    )

    print(
        f"Nulos tras imputar por promedio de nombre: "
        f"{df['nacimiento'].isnull().sum()}"
    )

    #4. Imputar restantes con promedio global
    promedio_global = round(df["nacimiento"].mean())

    df["nacimiento"] = df["nacimiento"].fillna(
        promedio_global
    )

    print(
        f"Nulos tras imputar por promedio global: "
        f"{df['nacimiento'].isnull().sum()}"
    )

    #5. Chequeo final
    print(
        f"Nulos restantes: "
        f"{df['nacimiento'].isnull().sum()}"
    )

    #6. Convertir a entero
    df["nacimiento"] = df["nacimiento"].astype(int)

    return df